# Smart Sales & Customer Analytics System
## Pandas Advanced Review Project

### Projenin Amacı
Bu projenin amacı; gerçek hayatta sıklıkla karşılaşılan **dağınık, eksik ve ilişkili verileri**
Pandas kullanarak temizlemek, birleştirmek ve anlamlı iş çıktıları üretmektir.

### Pandas Neden Tercih Edilir?
- Tablolu verilerle çalışma kolaylığı
- Eksik veri yönetimi
- Güçlü gruplama ve birleştirme işlemleri
- Excel ile doğrudan entegrasyon

### Kapsanan Pandas Konuları
- Eksik veriler (`isna`, `fillna`, `dropna`)
- GroupBy & Aggregation
- Merge & Concat
- İleri Pandas operasyonları
- Excel okuma & yazma

In [1]:
## 2. Veri Setlerini Oluşturma
import pandas as pd
import numpy as np

# Customers DataFrame
customers = pd.DataFrame({
    "CustomerID": [1, 2, 3, 4, 5, 6],
    "CustomerName": ["Ali", "Ayşe", "Mehmet", "Zeynep", "Can", "Elif"],
    "City": ["İstanbul", "Ankara", "İstanbul", "İzmir", "Ankara", "İstanbul"],
    "Age": [34, np.nan, 45, 29, np.nan, 41]
})

# Sales DataFrame
sales = pd.DataFrame({
    "SaleID": [101, 102, 103, 104, 105, 106],
    "CustomerID": [1, 2, 3, 1, 5, 6],
    "Product": ["Laptop", "Phone", "Tablet", "Phone", "Laptop", "Tablet"],
    "Amount": [12000, 8000, np.nan, 7500, 15000, 6000]
})

# Regions DataFrame
regions = pd.DataFrame({
    "City": ["İstanbul", "Ankara", "İzmir"],
    "Region": ["Marmara", "İç Anadolu", "Ege"]
})

customers, sales, regions

(   CustomerID CustomerName      City   Age
 0           1          Ali  İstanbul  34.0
 1           2         Ayşe    Ankara   NaN
 2           3       Mehmet  İstanbul  45.0
 3           4       Zeynep     İzmir  29.0
 4           5          Can    Ankara   NaN
 5           6         Elif  İstanbul  41.0,
    SaleID  CustomerID Product   Amount
 0     101           1  Laptop  12000.0
 1     102           2   Phone   8000.0
 2     103           3  Tablet      NaN
 3     104           1   Phone   7500.0
 4     105           5  Laptop  15000.0
 5     106           6  Tablet   6000.0,
        City      Region
 0  İstanbul     Marmara
 1    Ankara  İç Anadolu
 2     İzmir         Ege)

In [6]:
## 3. Veri Setlerini Oluşturma

# Eksik verilerin tespiti
customers.isna(), sales.isna()

(   CustomerID  CustomerName   City    Age
 0       False         False  False  False
 1       False         False  False  False
 2       False         False  False  False
 3       False         False  False  False
 4       False         False  False  False
 5       False         False  False  False,
    SaleID  CustomerID  Product  Amount
 0   False       False    False   False
 1   False       False    False   False
 3   False       False    False   False
 4   False       False    False   False
 5   False       False    False   False)

In [7]:
# Yaş bilgisini ortalama ile doldurma
average_age = customers["Age"].mean()
customers["Age"] = customers["Age"].fillna(average_age)

# Satış tutarı eksik olan kayıtları silme
sales = sales.dropna(subset=["Amount"])

customers, sales
# Burada amaç: her eksik veri aynı şekilde ele alınmaz.

(   CustomerID CustomerName      City    Age
 0           1          Ali  İstanbul  34.00
 1           2         Ayşe    Ankara  37.25
 2           3       Mehmet  İstanbul  45.00
 3           4       Zeynep     İzmir  29.00
 4           5          Can    Ankara  37.25
 5           6         Elif  İstanbul  41.00,
    SaleID  CustomerID Product   Amount
 0     101           1  Laptop  12000.0
 1     102           2   Phone   8000.0
 3     104           1   Phone   7500.0
 4     105           5  Laptop  15000.0
 5     106           6  Tablet   6000.0)

In [8]:
## 4. Merge İşlemleri

# Customers + Sales
customer_sales = pd.merge(customers, sales, on="CustomerID", how="inner")

# Regions ile birleştirme
full_data = pd.merge(customer_sales, regions, on="City", how="left")

full_data


,CustomerID,CustomerName,City,Age,SaleID,Product,Amount,Region
0,1,Ali,İstanbul,34.00,101,Laptop,12000.0,Marmara
1,1,Ali,İstanbul,34.00,104,Phone,7500.0,Marmara
2,2,Ayşe,Ankara,37.25,102,Phone,8000.0,İç Anadolu
3,5,Can,Ankara,37.25,105,Laptop,15000.0,İç Anadolu
4,6,Elif,İstanbul,41.00,106,Tablet,6000.0,Marmara


In [9]:
## 5. Gruplandırma (GroupBy)

# Şehir bazlı toplam satış
city_sales = full_data.groupby("City").agg(
    TotalSales=("Amount", "sum"),
    AverageSales=("Amount", "mean")
)

# Bölge bazlı ortalama satış
region_sales = full_data.groupby("Region").agg(
    AverageSales=("Amount", "mean")
)

# Ürün bazlı satış sayısı
product_sales_count = full_data.groupby("Product").agg(
    SalesCount=("SaleID", "count")
)

city_sales, region_sales, product_sales_count

(          TotalSales  AverageSales
 City                              
 Ankara       23000.0       11500.0
 İstanbul     25500.0        8500.0,
             AverageSales
 Region                  
 Marmara           8500.0
 İç Anadolu       11500.0,
          SalesCount
 Product            
 Laptop            2
 Phone             2
 Tablet            1)

In [10]:
## 6. Concat İşlemleri
# Yeni ay satışları
new_month_sales = pd.DataFrame({
    "SaleID": [107, 108],
    "CustomerID": [2, 4],
    "Product": ["Laptop", "Phone"],
    "Amount": [11000, 9000]
})

# Satışları birleştirme (satır bazlı)
all_sales = pd.concat([sales, new_month_sales], axis=0, ignore_index=True)

all_sales

,SaleID,CustomerID,Product,Amount
0,101,1,Laptop,12000.0
1,102,2,Phone,8000.0
2,104,1,Phone,7500.0
3,105,5,Laptop,15000.0
4,106,6,Tablet,6000.0
5,107,2,Laptop,11000.0
6,108,4,Phone,9000.0


In [11]:
## 7. Pandas İleri Operasyonlar
total_sales = full_data["Amount"].sum()
top_city = city_sales["TotalSales"].idxmax()
top_region = region_sales["AverageSales"].idxmax()

print(f"Toplam satış: {total_sales}")
print(f"En kârlı şehir: {top_city}")
print(f"En güçlü bölge: {top_region}")

Toplam satış: 48500.0
En kârlı şehir: İstanbul
En güçlü bölge: İç Anadolu


In [12]:
## 8. Excel ile Çalışma
# Excel çıktısı
full_data.to_excel("final_sales_report.xlsx", index=False)

In [13]:
## Yönetici Özeti
total_sales = full_data["Amount"].sum()
top_city = city_sales["TotalSales"].idxmax()
top_region = region_sales["AverageSales"].idxmax()

print(f"Toplam satış: {total_sales}")
print(f"En kârlı şehir: {top_city}")
print(f"En güçlü bölge: {top_region}")

Toplam satış: 48500.0
En kârlı şehir: İstanbul
En güçlü bölge: İç Anadolu
